# 01 — Arquitectura y servicios

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

| Grupo benchmark | Instancia | Diferenciación esperada |
|-----------------|-----------|-------------------------|
| 3D-BPP | `showcase_3d_bpp_instance.json` | best_fit ~27 emp. vs extreme_points ~22 |
| Híbrido (misma instancia 3D) | `showcase_3d_bpp_instance.json` | compaction/LS mueven 8–11 piezas |
| Mejora 3D (misma instancia 3D) | `showcase_3d_bpp_instance.json` | relocation/swap/orientation/bin_reduction refinan layout |
| Container Loading | `showcase_container_loading_instance.json` | weight_aware ~27 vs single_container ~17 |
| Single Container | `showcase_single_container_instance.json` | best_fit ~13 (94% util) vs first_fit ~16 |
| Cartonization | `showcase_cartonization_instance.json` | single-box vs multi_box |
| Palletization | `showcase_palletization_instance.json` | layer_based vs stack_based |
| Stacking-aware | `showcase_stacking_aware_instance.json` | stacking_aware vs stack_based |

Cada algoritmo implementado es un **servicio** con endpoint propio y contrato JSON homogéneo.


In [1]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


Proyecto: /home/edmundo/packing-services


In [2]:
from IPython.display import Markdown, display as ipy_display
from packing_services.services.metadata_service import MetadataService
from packing_services.benchmark.profiles import list_profiles

meta = MetadataService().get_metadata()
ipy_display(Markdown(f'## **{meta.service_name}** (v{meta.service_version})\n\n{meta.description}'))
for group in ('3D_BPP', '3D_HYBRID', 'CONTAINER_LOADING', 'CARTONIZATION', 'SINGLE_CONTAINER_LOADING'):
    display.show_comparable_algorithms(group)


## **packing-services** (v0.1.0)

Servicios modulares de Cutting and Packing. Incluye heurísticas constructivas, mejora local, metaheurísticas 3D-BPP, validador geométrico propio y comparador. Preparado para adaptadores a motores externos.

Motores comparables — 3D Bin Packing:


#,Algoritmo,Nombre,Opcional
1,heuristic_3d_bpp_v1,Volume First Candidate Placement,No
2,first_fit_decreasing_3d,First Fit Decreasing 3D,No
3,extreme_points_3d,Extreme Points Heuristic,No
4,best_fit_decreasing_3d,Best Fit Decreasing 3D,No
5,maximal_spaces_3d,Maximal Empty Spaces 3D,No
6,py3dbp_adapter,py3dbp Adapter (baseline externo),Sí


Motores comparables — 3D Bin Packing — Constructivo + mejora local:


#,Algoritmo,Nombre,Opcional
1,best_fit_decreasing_3d,Best Fit Decreasing 3D (baseline),No
2,solution_compaction,Compaction (mejora local),No
3,constructive_plus_local_search,Constructive + Local Improvement,No


...

#,Algoritmo,Nombre,Opcional
1,single_container_constructive,Single Container Constructive,No
2,weight_aware_container_loading,Weight-aware Container Loading,No
3,wall_building_3d,Wall-building Heuristic,No
4,extreme_points_3d,Extreme Points (multi-contenedor),No


Motores comparables — Cartonization:


#,Algoritmo,Nombre,Opcional
1,smallest_feasible_box,Smallest Feasible Box,No
2,best_box_volume_utilization,Best Box by Volume Utilization,No
3,first_fit_box,First Fit Box (orden catálogo),No
4,largest_feasible_box,Largest Feasible Box,No
5,multi_box_cartonization,Multi-box Cartonization,No


Motores comparables — Single Container Loading:


#,Algoritmo,Nombre,Opcional
1,single_container_constructive,Single Container Constructive,No
2,best_fit_decreasing_3d,Best Fit Decreasing 3D,No
3,first_fit_decreasing_3d,First Fit Decreasing 3D,No


In [3]:
import pandas as pd
from IPython.display import display as ipy_display
from packing_services.benchmark.profiles import list_profiles

endpoints = [
    ('Ejecutar algoritmo', 'POST /algorithms/{algorithm_name}/execute', 'Input normalizado por tipo; nombre en URL (A3)'),
    ('3D Bin Packing (legacy)', 'POST /pack/3d-bpp', 'Empaquetar en contenedores'),
    ('Benchmark', 'POST /benchmark', 'Comparar motores (perfil o engines explícitos)'),
    ('Perfiles benchmark', 'GET /benchmark/profiles', 'Catálogo de perfiles estándar'),
    ('Container Loading (legacy)', 'POST /pack/container-loading', 'Carga multi-contenedor + peso'),
    ('Cartonization (legacy)', 'POST /pack/cartonization', 'Elegir caja + layout'),
    ('Validation', 'POST /validate', 'Juez independiente'),
    ('Dataspace', 'GET /services', 'Descriptores publicables'),
]
ipy_display(pd.DataFrame(endpoints, columns=['Servicio', 'Endpoint', 'Rol']).style.hide(axis='index'))
profiles = list_profiles()
print(f'Perfiles de benchmark disponibles: {len(profiles)}')
ipy_display(pd.DataFrame([{'Tipo': p['problem_type'], 'Perfil': p['profile'], 'Motores': p['engines_count']} for p in profiles]).style.hide(axis='index'))
from _shared.loaders import SHOWCASE_DIFFERENTIATION, SHOWCASE_INSTANCES, BENCHMARK_GROUPS
showcase_rows = [{'Grupo': g, 'Instancia': SHOWCASE_INSTANCES.get(g, '—'), 'Diferenciación': SHOWCASE_DIFFERENTIATION.get(g, '—')} for g in BENCHMARK_GROUPS]
ipy_display(pd.DataFrame(showcase_rows).style.hide(axis='index'))


Servicio,Endpoint,Rol
Ejecutar algoritmo,POST /algorithms/{algorithm_name}/execute,Input normalizado por tipo; nombre en URL (A3)
3D Bin Packing (legacy),POST /pack/3d-bpp,Empaquetar en contenedores
Benchmark,POST /benchmark,Comparar motores (perfil o engines explícitos)
Perfiles benchmark,GET /benchmark/profiles,Catálogo de perfiles estándar
Container Loading (legacy),POST /pack/container-loading,Carga multi-contenedor + peso
Cartonization (legacy),POST /pack/cartonization,Elegir caja + layout
Validation,POST /validate,Juez independiente
Dataspace,GET /services,Descriptores publicables


Perfiles de benchmark disponibles: 11


Tipo,Perfil,Motores
3D_BPP,constructive,6
3D_BPP,hybrid,3
3D_BPP,improvement,4
3D_BPP,metaheuristic,4
CARTONIZATION,box_selection,4
CARTONIZATION,multi_box,3
CONTAINER_LOADING,constructive,4
CONTAINER_LOADING,improvement,3
PALLETIZATION,constructive,2
SINGLE_CONTAINER_LOADING,constructive,3


Grupo,Instancia,Diferenciación
3D_BPP,showcase_3d_bpp_instance,best_fit ~27 emp. (85% util) vs extreme_points ~22 (66%)
3D_HYBRID,showcase_3d_bpp_instance,misma cantidad empacada; compaction/LS mueven 8–11 piezas
3D_IMPROVEMENT,showcase_3d_bpp_instance,misma baseline; relocation/swap/orientation/bin_reduction refinan layout
3D_METAHEURISTIC,showcase_3d_bpp_instance,metaheurísticas optimizan orden de colocación sobre baseline constructivo
CONTAINER_LOADING,showcase_container_loading_instance,weight_aware ~27 emp. vs wall_building ~24 vs single_container ~17
CARTONIZATION,showcase_cartonization_instance,single-box → BOX_M (~49%) vs multi_box → varias cajas S/M
SINGLE_CONTAINER_LOADING,showcase_single_container_instance,best_fit/single ~13 emp. (94% util) vs first_fit ~16 (89%)
PALLETIZATION,showcase_palletization_instance,layer_based ~capas horizontales vs stack_based ~columnas
STACKING_AWARE,showcase_stacking_aware_instance,stacking_aware respeta soporte/carga vs stack_based libre


In [4]:
from _shared.loaders import build_algorithm_execute_input

# Ejemplo: un algoritmo por URL, body sin campo "algorithm"
algo = 'best_fit_decreasing_3d'
payload = build_algorithm_execute_input(algo, problem_type='3D_BPP')
print(f'POST /api/v1/algorithms/{algo}/execute')
print('Campos del body:', sorted(payload.keys()))
result = runners.run_algorithm_execute(algo, payload)
display.show_execute_result(result)


2026-07-13 12:46:24,378 | INFO    | packing_services.services.algorithm_execution_service | algorithm-execute name=best_fit_decreasing_3d problem_type=3D_BPP request_id=showcase-3d-bpp-001


2026-07-13 12:46:24,379 | INFO    | packing_services.services.packing_service | pack request_id=showcase-3d-bpp-001 algorithm=best_fit_decreasing_3d items=30 containers=2


2026-07-13 12:46:24,413 | INFO    | packing_services.services.packing_service | pack done request_id=showcase-3d-bpp-001 status=SolutionStatus.PARTIAL packed=27 unpacked=3 util=0.848


POST /api/v1/algorithms/best_fit_decreasing_3d/execute
Campos del body: ['constraints', 'containers', 'items', 'objective', 'parameters', 'problem_type', 'request_id']


PackingSolution(request_id='showcase-3d-bpp-001', status=<SolutionStatus.PARTIAL: 'partial'>, problem_type=<ProblemType.THREE_D_BPP: '3D_BPP'>, algorithm_name='best_fit_decreasing_3d', packed_items=[PackedItem(item_id='P2#1', container_id='C1', position=Point3D(x=0.0, y=0.0, z=0.0), orientation=Orientation(length=60.0, width=30.0, height=40.0), weight=7.0), PackedItem(item_id='P2#2', container_id='C1', position=Point3D(x=0.0, y=0.0, z=40.0), orientation=Orientation(length=60.0, width=30.0, height=40.0), weight=7.0), PackedItem(item_id='P2#3', container_id='C1', position=Point3D(x=60.0, y=0.0, z=0.0), orientation=Orientation(length=40.0, width=30.0, height=60.0), weight=7.0), PackedItem(item_id='P2#4', container_id='C2', position=Point3D(x=0.0, y=0.0, z=0.0), orientation=Orientation(length=60.0, width=30.0, height=40.0), weight=7.0), PackedItem(item_id='P2#5', container_id='C2', position=Point3D(x=0.0, y=0.0, z=40.0), orientation=Orientation(length=60.0, width=30.0, height=40.0), weight

**Siguiente:** demo visual 3D-BPP → `02_demo_3d_bin_packing.ipynb`
